# 1 · Data Exploration — BWI Flights + Weather

Before modeling anything, you want to **understand the data**: what it contains, what the target looks like, and which features plausibly drive delays.

This notebook walks through:
1. Schema and summary stats
2. Target distributions (delay, cancellation)
3. Temporal patterns (hour of day, day of week)
4. Categorical effects (by carrier, destination)
5. Weather effects (precipitation, wind, temperature)
6. Correlations between numeric features

> **Your job as an analyst**: for each chart, write one sentence explaining what you see and whether it matches your intuition.

## Setup

**Required packages**
```
pip install pandas numpy matplotlib seaborn scikit-learn xgboost
```
The dataset is `bwi_model_ready.csv` — already cleaned by `preprocess.py`.
It covers 14 days at Baltimore/Washington International (BWI), joined with hourly weather.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

df = pd.read_csv("bwi_model_ready.csv")
print("shape:", df.shape)
df.head()

## 1. Schema and summary

Start by looking at column types, missingness, and numeric ranges.

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
# Missingness per column
df.isna().mean().sort_values(ascending=False).to_frame("null_rate").head(10)

## 2. Target distributions

We have **three targets**:

- `is_cancelled`  — was the flight cancelled? (very rare)
- `arrival_delay_minutes` — by how many minutes did it arrive late? (negative = early)
- `is_delayed` — shorthand for `arrival_delay_minutes >= 15`

Class balance matters for choosing metrics later.

In [ ]:
targets = pd.DataFrame({
    'is_cancelled':    [df['is_cancelled'].mean()],
    'is_delayed >=15': [df['is_delayed'].mean()],
})
targets.T.rename(columns={0: 'positive rate'}).style.format('{:.2%}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df['arrival_delay_minutes'].clip(-60, 180).hist(bins=60, ax=ax[0])
ax[0].axvline(15, color='red', ls='--', label='delay threshold = 15 min')
ax[0].set(title='Arrival delay (min)', xlabel='minutes', ylabel='flights')
ax[0].legend()

df['is_delayed'].value_counts().plot.bar(ax=ax[1])
ax[1].set(title='Delayed vs on-time', xlabel='is_delayed', ylabel='flights')
plt.tight_layout(); plt.show()

**Questions to answer in your writeup**
- What does the shape of the delay-minutes histogram tell you?
- Is the `is_delayed` target balanced or imbalanced? How will that affect metric choice?

## 3. Temporal patterns

Delays tend to compound through the day. Do they here?

In [ ]:
by_hour = df.groupby('dep_hour')['is_delayed'].agg(['mean', 'count'])
fig, ax = plt.subplots(figsize=(10, 4))
by_hour['mean'].plot.bar(ax=ax, color='steelblue')
ax.set(title='Delay rate by departure hour (UTC)', ylabel='P(delayed)', xlabel='hour')
plt.tight_layout(); plt.show()

by_hour.head()

In [ ]:
by_dow = df.groupby('departure_day_of_week')['is_delayed'].mean()
ax = by_dow.plot.bar(color='teal')
ax.set(title='Delay rate by day of week (1=Mon .. 7=Sun)',
       ylabel='P(delayed)', xlabel='day of week')
plt.tight_layout(); plt.show()

## 4. Categorical effects

Which carriers and destinations are most delay-prone?

In [ ]:
by_carrier = (df.groupby('carrier', observed=True)
                .agg(n=('is_delayed', 'size'),
                     delay_rate=('is_delayed', 'mean'))
                .sort_values('delay_rate', ascending=False))
by_carrier.head(15)

In [ ]:
top = by_carrier[by_carrier['n'] >= 50].sort_values('delay_rate')
ax = top['delay_rate'].plot.barh(color='indianred')
ax.set(title='Delay rate by carrier (n ≥ 50 flights)', xlabel='P(delayed)')
plt.tight_layout(); plt.show()

In [ ]:
by_dest = (df.groupby('destination_airport', observed=True)
             .agg(n=('is_delayed', 'size'),
                  delay_rate=('is_delayed', 'mean'))
             .query('n >= 50')
             .sort_values('delay_rate', ascending=False)
             .head(15))
by_dest

## 5. Weather effects

Intuition says precipitation, wind, and low visibility all push delays up. Let's check by bucketing each variable and plotting the delay rate.

In [ ]:
def bucket_delay_rate(col, bins, labels=None):
    tmp = df.copy()
    tmp['bucket'] = pd.cut(tmp[col], bins=bins, labels=labels, include_lowest=True)
    return tmp.groupby('bucket', observed=True)['is_delayed'].agg(['mean', 'count'])

precip = bucket_delay_rate('dep_weather_precip_mm',
                           bins=[-0.01, 0.0, 0.5, 2.0, 5.0, 100],
                           labels=['none', '<0.5mm', '0.5–2mm', '2–5mm', '>5mm'])
precip

In [ ]:
wind = bucket_delay_rate('dep_weather_wind_kph',
                         bins=[-1, 5, 10, 20, 30, 200],
                         labels=['<5', '5–10', '10–20', '20–30', '>30'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
precip['mean'].plot.bar(ax=axes[0], color='royalblue')
axes[0].set(title='Delay rate vs precipitation', ylabel='P(delayed)', xlabel='precip (mm/h)')
wind['mean'].plot.bar(ax=axes[1], color='slateblue')
axes[1].set(title='Delay rate vs wind speed', ylabel='P(delayed)', xlabel='wind (kph)')
plt.tight_layout(); plt.show()

In [ ]:
# Scatter: delay minutes vs precipitation (log-ish scale, jittered)
sub = df.dropna(subset=['arrival_delay_minutes']).sample(min(2000, len(df)), random_state=0)
plt.scatter(sub['dep_weather_precip_mm'] + 0.01, sub['arrival_delay_minutes'],
            alpha=0.3, s=10)
plt.xscale('log')
plt.axhline(15, color='red', ls='--', label='delay threshold')
plt.xlabel('precipitation (mm/h, log)')
plt.ylabel('arrival delay (min)')
plt.title('Precip vs arrival delay')
plt.legend(); plt.tight_layout(); plt.show()

## 6. Correlation heatmap

Correlation between numeric features helps you spot redundancy (e.g., temperature and dew point), and catch leakage if something correlates too strongly with the target.

In [ ]:
num = df.select_dtypes(include='number').drop(columns=['is_cancelled'])
corr = num.corr()
import seaborn as sns
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, ax=ax,
            cbar_kws={'shrink': 0.7})
ax.set_title('Feature correlations')
plt.tight_layout(); plt.show()

## 7. Writing up your findings

Before you move on to modelling, put the answers to these in a short writeup:
1. **Baseline**: if a naive model predicted "on-time" for every flight, what would its accuracy be?
2. **Imbalance**: which metric (accuracy, ROC AUC, PR AUC, F1) will be most informative given the class balance?
3. **Expected drivers**: based on what you saw, which 3–4 features do you expect the model to rely on most?
4. **Watch-outs**: any columns you'd be worried might leak information about the target?